# Fine-tune จับทรงกล่องจากไดไลน์ v2.2 (Colab GPU)

**EfficientNet-B3** + eval บน adjudicated holdout (per-case audit)

**⚠️ แก้ตาม GPT Round 16:**
- **11-class active** (`ACTIVE=[1,2,3,4,5,7,8,9,10,11,12]`) — **class 6 (Frame-Vue) = 0 ตัวอย่าง ตัดออก** → ไม่ใช่ 12-way
- **mixed-label families ตัดทิ้งใน prep** (6 family ที่ label ขัด) + crop bbox (feature เล็กได้ pixel มากขึ้น)
- **group-STRATIFIED split** (StratifiedGroupKFold) + assert ทุก active class มี family สองฝั่ง
- **D4 augmentation** (0/90/180/270 + reflection, fill=255) — train เท่านั้น, invariance เต็ม
- metrics: **macro-F1 / balanced-acc / per-class recall / confusion** (accuracy รวมถูกครอบโดย class 1-4 cap 500)
- multi-seed [42,1,7] + best-checkpoint จาก proxy-val (ไม่แตะ holdout)
- holdout 9 ใบ = expert 4 + claude 5, ครอบแค่ class 1,2,9,12 → **per-case audit ไม่ใช่ accuracy**
- baseline B0 51%/CLIP 43% = historical (คนละ split ไม่ head-to-head)

**ขั้นตอน:** Runtime→GPU(T4) → Run all → อัป `dieline_train.zip` (ตัวใหม่)


In [ ]:
!pip -q install timm scikit-learn


### 1. อัปโหลด dieline_train.zip


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up.keys())[0]).extractall('data')
print('train:', len(os.listdir('data/img')), '| gold:', len(os.listdir('data/gold')))


### 2. โหลด labels + group-split (train/val ภายใน)


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
ACTIVE=[1,2,3,4,5,7,8,9,10,11,12]                    # class 6 (Frame-Vue)=0 ตัวอย่าง → ตัด (ไม่ใช่ 12-way)
cls2idx={c:i for i,c in enumerate(ACTIVE)}; idx2cls={i:c for c,i in cls2idx.items()}; NC=len(ACTIVE)
df=pd.read_csv('data/labels.csv'); df=df[df.label.isin(ACTIVE)].reset_index(drop=True); df['y']=df.label.map(cls2idx)
gold=pd.read_csv('data/gold_labels.csv'); gold['y']=gold.label.map(cls2idx)
# group-STRATIFIED split (group-safe + stratify class) — fallback เป็น GroupShuffleSplit ถ้า rare class ทำ error
try:
    tr,va=next(StratifiedGroupKFold(n_splits=6,shuffle=True,random_state=42).split(df,df.y,groups=df.group_id))
    split='StratifiedGroupKFold'
except Exception as e:
    print('StratifiedGroupKFold ไม่ได้ (%s) → ใช้ GroupShuffleSplit'%type(e).__name__)
    tr,va=next(GroupShuffleSplit(1,test_size=0.15,random_state=42).split(df,groups=df.group_id)); split='GroupShuffleSplit'
train_df,val_df=df.iloc[tr].reset_index(drop=True),df.iloc[va].reset_index(drop=True)
assert not (set(train_df.group_id)&set(val_df.group_id)), 'val group leak'
famc=lambda d,c: d[d.label==c].group_id.nunique()
print('split=%s | class : train_rows/fam | val_rows/fam'%split)
miss=[]
for c in ACTIVE:
    tr_r,va_r=(train_df.label==c).sum(),(val_df.label==c).sum(); ftr,fva=famc(train_df,c),famc(val_df,c)
    ok=ftr>0 and fva>0;  miss+= [] if ok else [c]
    print('  %2d : %4d/%3d | %3d/%3d%s'%(c,tr_r,ftr,va_r,fva,'' if ok else '  ⚠️ไม่ครบสองฝั่ง (ประเมินคลาสนี้ไม่ได้)'))
print('train',len(train_df),'| val',len(val_df),'| active',NC,'classes (ตัด 6)','| ⚠️คลาสประเมินไม่ได้:',miss or 'ไม่มี')
print('holdout source:',gold.source.value_counts().to_dict(),'| ต่อทรง:',gold.label.value_counts().sort_index().to_dict())


### 3. Dataset


In [ ]:
import torch, timm, numpy as np, torch.nn as nn, random
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
SZ=320
NORM=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
class D4:  # invariance เต็ม: rotation 0/90/180/270 + reflection | fill ขาว (พื้นกระดาษ) | train เท่านั้น
    def __call__(s,im):
        im=TF.rotate(im,90*random.randint(0,3),fill=255)
        if random.random()<0.5: im=TF.hflip(im)
        return im
aug=T.Compose([T.Resize((SZ,SZ)),D4(),T.RandomRotation(3,fill=255),T.ColorJitter(0.05,0.05),T.ToTensor(),NORM])
plain=T.Compose([T.Resize((SZ,SZ)),T.ToTensor(),NORM])   # val/gold: deterministic ห้าม augment
class DS(Dataset):
    def __init__(s,d,folder,tf): s.d=d; s.f=folder; s.tf=tf
    def __len__(s): return len(s.d)
    def __getitem__(s,i):
        r=s.d.iloc[i]; im=Image.open(f'data/{s.f}/{r.file}').convert('RGB')
        return s.tf(im), int(r.y)
tl=DataLoader(DS(train_df,'img',aug),batch_size=24,shuffle=True,num_workers=2)
vl=DataLoader(DS(val_df,'img',plain),batch_size=48,num_workers=2)
gl=DataLoader(DS(gold,'gold',plain),batch_size=16)


### 4. เทรน (EfficientNet-B3, 11-class active, class-weighted, multi-seed + best-checkpoint)
`num_classes=NC`(11) · D4 aug (cell 3) · seed ครบ · best_state จาก **proxy-val** (ไม่แตะ holdout) · 3 seeds mean/range


In [ ]:
import random, statistics
dev='cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
cnt=train_df.y.value_counts().sort_index()
w=torch.tensor([(len(train_df)/(NC*cnt.get(i,1)))**0.5 for i in range(NC)],dtype=torch.float).clamp(max=4).to(dev)  # sqrt+cap, active NC

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def val_acc(m):
    m.eval(); c=n=0
    with torch.no_grad():
        for x,y in vl: c+=(m(x.to(dev)).argmax(1).cpu()==y).sum().item(); n+=len(y)
    return 100*c/n

def train_one(seed, epochs=20):
    seed_all(seed)
    m=timm.create_model('efficientnet_b3',pretrained=True,num_classes=NC).to(dev)
    crit=nn.CrossEntropyLoss(weight=w,label_smoothing=0.05)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=3e-4,total_steps=epochs*len(tl),pct_start=0.15)
    best_val=-1; best_state=None
    for ep in range(epochs):
        m.train(); tot=0
        for x,y in tl:
            x,y=x.to(dev),y.to(dev); opt.zero_grad(); loss=crit(m(x),y); loss.backward(); opt.step(); sch.step(); tot+=loss.item()
        va=val_acc(m)
        if va>best_val: best_val=va; best_state={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
        print('  seed%d ep%2d loss%.3f val %.1f%% (best %.1f%%)'%(seed,ep+1,tot/len(tl),va,best_val))
    m.load_state_dict(best_state)
    return m,best_val

SEEDS=[42,1,7]
runs=[]; best=(-1,None,None)
for s in SEEDS:
    m,bv=train_one(s); runs.append((s,bv))
    if bv>best[0]: best=(bv,m,s)
print('\n=== proxy-val best/seed (group-STRATIFIED) — B0 51% historical (คนละ split ไม่ head-to-head) ===')
for s,bv in runs: print('  seed %d: %.1f%%'%(s,bv))
vals=[bv for _,bv in runs]
print('  mean %.1f%% | range %.1f–%.1f%%'%(statistics.mean(vals),min(vals),max(vals)))
model=best[1]; print('>> เลือก seed %d (val %.1f%%) → ประเมิน holdout (เลือกจาก proxy-val เท่านั้น)'%(best[2],best[0]))


### 5. ประเมินบน proxy val (เทียบ baseline)


In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, confusion_matrix
model.eval(); Pi=[]; Yi=[]
with torch.no_grad():
    for x,y in vl: Pi+=model(x.to(dev)).argmax(1).cpu().tolist(); Yi+=y.tolist()
Pi,Yi=np.array(Pi),np.array(Yi)
P=np.array([idx2cls[i] for i in Pi]); Y=np.array([idx2cls[i] for i in Yi])   # idx → class จริง
print('=== proxy val (noisy label) — active %d classes ==='%NC)
print('accuracy: %.1f%%  (ถูกครอบโดย class 1-4 cap 500 → ดู macro/per-class ด้วย)'%(100*(Pi==Yi).mean()))
print('balanced-acc: %.1f%%  |  macro-F1: %.3f'%(100*balanced_accuracy_score(Yi,Pi),f1_score(Yi,Pi,average='macro')))
tk=np.isin(Y,[1,2,4,11]); print('tuck-family (1/2/4/11): %.0f%% (n=%d)'%(100*(P[tk]==Y[tk]).mean(),tk.sum()))
print('custom-vs-std: %.0f%%'%(100*((P==12)==(Y==12)).mean()))
print('\nper-class recall/precision/support:')
print(classification_report(Y,P,labels=ACTIVE,digits=2,zero_division=0))
print('confusion (แถว=จริง, คอลัมน์=ทาย) labels=%s:'%ACTIVE); print(confusion_matrix(Y,P,labels=ACTIVE))
print('\n(baseline B0 51% / CLIP 43% = historical คนละ split — ยังไม่ head-to-head)')


### 6. 🎯 ประเมินบน adjudicated holdout — **per-case AUDIT** (แยก expert vs claude)
⚠️ ไม่ใช่ "12-way accuracy" (ครอบแค่ 4 class) — ดูว่าโมเดล **แก้** custom ที่ proxy ผิด จริงไหม


In [ ]:
model.eval(); GPi=[]
with torch.no_grad():
    for x,y in gl: GPi+=model(x.to(dev)).argmax(1).cpu().tolist()
g=gold.copy(); g['pred']=[idx2cls[i] for i in GPi]; g['ok']=g['pred']==g['label']   # idx → class จริง
print('=== adjudicated holdout (%d ใบ) — per-case AUDIT (ไม่ใช่ 12-way accuracy) ==='%len(g))
for src in ['expert','claude']:
    d=g[g.source==src]
    if len(d): print('  %-7s n=%d: %d/%d ถูก'%(src,len(d),int(d.ok.sum()),len(d)))
print('  combined n=%d: %d/%d = %.0f%%  (Claude/CV บนชุดนี้ = 44%%)'%(len(g),int(g.ok.sum()),len(g),100*g.ok.mean()))
print('  ต่อใบ (label[src] -> ทาย):')
for _,r in g.iterrows(): print('    %2d[%s] -> %2d %s'%(r.label,r.source[:3],r.pred,'✓' if r.ok else '✗'))
cust=g[g.label==12]
print('  >> custom(12) ที่ proxy มัก under-label: %d/%d โมเดลตีถูก'%(int(cust.ok.sum()),len(cust)))
print('  ⚠️ ครอบเพียง class %s → ห้ามสรุป "B3 12-way ชนะ B0" จากชุดนี้'%sorted(g.label.unique()))


### 7. เซฟโมเดล


In [ ]:
torch.save(model.state_dict(),'dieline_effb3.pt')
from google.colab import files; files.download('dieline_effb3.pt')
